# Transfer learning and fine-tuning — ResNet50

**DeepVision Masterclass — Modern Edition**

This notebook is designed as an executable teaching lab. Read the dataset contract and validation notes before training.

## 1. Dataset contract
This notebook uses CIFAR-10 resized to 224×224 as a quick public demonstration. For the original Homer/Bart exercise, point `ImageFolder`/`image_dataset_from_directory` to a legally obtained archive and preserve the class-to-index mapping.

In [ ]:
import torch, torchvision, tensorflow as tf
print('torch',torch.__version__,'torchvision',torchvision.__version__,'tensorflow',tf.__version__)

## 2. PyTorch: pretrained ResNet50
Torchvision weights include their recommended preprocessing. Replace the classification head and freeze the backbone first.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
weights=ResNet50_Weights.DEFAULT
preprocess=weights.transforms()
model_t=resnet50(weights=weights)
for p in model_t.parameters(): p.requires_grad=False
model_t.fc=torch.nn.Linear(model_t.fc.in_features,10)
for p in model_t.fc.parameters(): p.requires_grad=True
print('Trainable:',sum(p.numel() for p in model_t.parameters() if p.requires_grad))

## 3. Fine-tune in two phases
Train the head, then unfreeze only the final residual stage and use a smaller learning rate. Use a validation split for early stopping.

In [ ]:
# Example phase switch after training the classifier head:
for p in model_t.layer4.parameters(): p.requires_grad=True
# Rebuild optimizer after changing trainable parameters:
optimizer_t=torch.optim.AdamW(filter(lambda p:p.requires_grad,model_t.parameters()),lr=1e-5,weight_decay=1e-4)
# IMPORTANT: use ImageNet normalization and the exact preprocessing for these weights.

## 4. TensorFlow/Keras: pretrained ResNet50
Keras uses `preprocess_input` (not a generic 0–1 rescale) for the ResNet50 application.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
base=keras.applications.ResNet50(include_top=False,weights='imagenet',input_shape=(224,224,3))
base.trainable=False
inputs=keras.Input((224,224,3))
x=keras.applications.resnet50.preprocess_input(inputs)
x=base(x,training=False)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dropout(.25)(x)
outputs=layers.Dense(10,activation='softmax')(x)
model_k=keras.Model(inputs,outputs)
model_k.compile(optimizer=keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model_k.summary()

## 5. Fine-tuning safely
Unfreeze the final layers only after the head converges. Keep BatchNorm in inference mode to avoid corrupting pretrained statistics on small datasets.

In [ ]:
base.trainable=True
for layer in base.layers[:-30]: layer.trainable=False
for layer in base.layers:
    if isinstance(layer,keras.layers.BatchNormalization): layer.trainable=False
model_k.compile(optimizer=keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
# model_k.fit(train_ds,validation_data=val_ds,epochs=5,callbacks=[keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)])

## 6. Save portable artifacts
PyTorch state dictionaries and Keras `.keras` archives are preferred over serializing arbitrary Python objects.

In [ ]:
torch.save(model_t.state_dict(),'resnet50_transfer_torch.pt')
# model_k.save('resnet50_transfer_keras.keras')
# Store class_names, preprocessing, input size, library versions and evaluation protocol beside the model.

## 7. Exercises
Compare frozen features vs fine-tuning; test EfficientNetV2/MobileNetV3; compare latency and parameter count; avoid claiming transfer-learning gains without a controlled split.